# Fresh approach: DINOv2 foundation model for rice leaf disease

Everything so far used a CNN trained on ImageNet labels. This notebook uses **DINOv2 ViT-S/14**, a vision transformer that learned general visual features from 142 million images without labels.

| Question | Test | Compared with |
|---|---|---|
| Is it more accurate? | Same folds and hold-out as before | Flat ResNeXt-50 answers stored in this notebook |
| Does it survive a new source of close-ups? | Train on `named`, test on `r6k` | A ResNeXt-50 trained the same way here |
| Does it make different mistakes? | Overlap with the old models' errors | Flat and hierarchical answers stored in this notebook |

Two versions are tested: **frozen** (DINOv2 unchanged, with a simple linear classifier on top) and **fine-tuned** (last two transformer blocks retrained).

**Honesty note:** these images were seen in earlier experiments, but DINOv2 has not been tuned on any of them. Report the results as they come out.

**What it needs:** only the dataset, `processed.zip`. The folds and the old models' answers are stored inside this notebook. The notebook detects Colab or Kaggle by itself, so there is nothing to change. Expect about 20 minutes on either.

**To run on Colab:**
1. Put `processed.zip` in `MyDrive/rldd`.
2. Choose a **T4 GPU**, then **Run all**.
3. Send the zip it downloads. If the session drops, Run all again: finished steps are reloaded from Drive.

**To run on Kaggle:**
1. Upload `processed.zip` as a Kaggle dataset and add it with **Add Input**. It works whether Kaggle keeps the zip or unpacks it.
2. In **Settings**, choose **Accelerator: GPU T4 x2** and turn **Internet** on. Internet needs a phone-verified Kaggle account.
3. Choose **Run All**.
4. Download `fresh_dino_results_for_review.zip` from the link at the end, or from the **Output** panel, and send it. Kaggle clears its files when the session ends, so download it before closing.

## 1. Setup and fixed settings

In [1]:
!pip -q install timm || echo 'timm could not be installed: on Kaggle, turn Internet on in Settings'

In [2]:
import os, re, gc, io, time, json, copy, zipfile, base64, zlib, random, hashlib, shutil, tempfile, urllib.request, warnings
os.environ['HF_HUB_DISABLE_XET'] = '1'
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
import timm
from PIL import Image
from IPython.display import display
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score
warnings.filterwarnings('ignore')

KAGGLE_INPUT = os.environ.get('CAHSR_KAGGLE_INPUT', '/kaggle/input')
IN_KAGGLE = os.path.isdir(KAGGLE_INPUT)
files, IN_COLAB = None, False
if IN_KAGGLE:
    BASE = os.environ.get('CAHSR_KAGGLE_WORK', '/kaggle/working')
else:
    try:
        from google.colab import drive, files
        drive.mount('/content/drive')
        BASE, IN_COLAB = '/content/drive/MyDrive/cashr', True
    except ImportError:
        BASE = os.environ['CAHSR_BASE']
print('running on', 'Kaggle' if IN_KAGGLE else 'Colab' if IN_COLAB else 'a local machine')

SMOKE = os.environ.get('CAHSR_SMOKE') == '1'    # tiny CPU check of the code, not an experiment
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
assert DEVICE == 'cuda' or SMOKE, 'No GPU found. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Then run all again.'
GPU_NAME = torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'cpu'
print('device:', GPU_NAME)
if 'P100' in GPU_NAME:
    print('NOTICE: Kaggle gave a P100, which recent PyTorch may not support. If you see a CUDA error, choose GPU T4 x2 instead.')
torch.backends.cudnn.benchmark = True

CFG = dict(
    zip_path='/content/drive/MyDrive/rldd/processed.zip', data_root='/content/data',
    dino='vit_small_patch14_dinov2.lvd142m', img_size=224, split_seed=2026, n_folds=3, val_splits=5, batch_size=32, num_workers=2,
    linear_weight_decays=[1e-4, 1e-3, 1e-2, 1e-1], linear_steps=500, linear_lr=1e-3,      # frozen version
    ft_blocks=2, ft_lr_backbone=2e-5, ft_lr_head=1e-3, ft_epochs=8, ft_patience=3, weight_decay=1e-4,   # fine-tuned version
    resnext='resnext50_32x4d', resnext_lr=3e-4, resnext_epochs=15, resnext_patience=4,    # the flat recipe, for the new-source test
    pretrained=True, n_boot=2000, n_perm=100000, rule_p=0.05, expected_val_sizes=[520, 527, 460], smoke=SMOKE,
)
if IN_KAGGLE:
    CFG.update(zip_path=None, data_root=os.environ.get('CAHSR_DATA', os.path.join(tempfile.gettempdir(), 'rice_data')))
elif not IN_COLAB:
    CFG.update(zip_path=os.path.expanduser('~/rldd/processed.zip'), data_root=os.environ['CAHSR_DATA'])
if SMOKE:
    CFG.update(img_size=70, batch_size=16, num_workers=0, pretrained=False, linear_steps=20, ft_epochs=1, ft_patience=1, resnext='resnet18', resnext_epochs=1, resnext_patience=1, n_boot=20, n_perm=200)
OUT = f'{BASE}/runs_fresh_dino'
PRED_DIR = f'{OUT}/preds'
os.makedirs(PRED_DIR, exist_ok=True)
SESSION_START = time.time()

def seed_everything(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

CLASSES = ['bacterial_leaf_blight', 'brown_spot', 'healthy', 'leaf_blast', 'leaf_scald', 'narrow_brown_spot']
NC, K, H_IDX = len(CLASSES), CFG['n_folds'], 2

running on Kaggle


AssertionError: No GPU found. Colab: Runtime > Change runtime type > T4 GPU. Kaggle: Settings > Accelerator > GPU T4 x2. Then run all again.

### 1.1 Pretrained weights
Downloaded once from the addresses timm lists, not from Hugging Face, and kept in `MyDrive/cashr/pretrained`. Failed downloads fall back to Hugging Face with a printed notice.

In [ ]:
WEIGHTS_DIR = f'{BASE}/pretrained'

def fetch_weights(name, url=None, tries=3):
    url = url or timm.create_model(name, pretrained=False).pretrained_cfg['url']
    fn = os.path.basename(url)
    dest = os.path.join(WEIGHTS_DIR, fn)
    os.makedirs(WEIGHTS_DIR, exist_ok=True)
    if os.path.exists(dest):
        print(f'{name}: using saved copy')
        return dest
    if IN_KAGGLE:      # a copy added as a Kaggle input also works, and needs no internet
        for root, _, names in os.walk(KAGGLE_INPUT, followlinks=True):
            if fn in names:
                print(f'{name}: using Kaggle input copy')
                return os.path.join(root, fn)
    m = re.search(r'-([0-9a-f]{8})\.pth$', fn)
    tmp = os.path.join(tempfile.gettempdir(), fn)
    for attempt in range(1, tries + 1):
        try:
            t0, sha, got = time.time(), hashlib.sha256(), 0
            with urllib.request.urlopen(url, timeout=30) as r, open(tmp, 'wb') as f:
                size = r.headers.get('Content-Length')
                while True:
                    b = r.read(1 << 20)
                    if not b: break
                    f.write(b); sha.update(b); got += len(b)
            assert size is None or got == int(size), 'incomplete download'
            assert not m or sha.hexdigest().startswith(m.group(1)), 'checksum mismatch'
            shutil.copy(tmp, dest + '.tmp'); os.replace(dest + '.tmp', dest)
            print(f'{name}: {got / 1e6:.0f} MB in {time.time() - t0:.0f} s')
            return dest
        except Exception as e:
            print(f'{name}: download attempt {attempt} failed ({e})')
    print(f'NOTICE: {name} could not be fetched directly, so Hugging Face will be used.' + (' On Kaggle, check that Internet is on in Settings.' if IN_KAGGLE else ''))
    return None

WEIGHT_FILES = {n: fetch_weights(n) for n in (CFG['dino'], CFG['resnext'])} if CFG['pretrained'] else {}

def overlay(name):
    return dict(file=WEIGHT_FILES[name]) if WEIGHT_FILES.get(name) else None

def make_dino():
    return timm.create_model(CFG['dino'], pretrained=CFG['pretrained'], img_size=CFG['img_size'], num_classes=0, pretrained_cfg_overlay=overlay(CFG['dino'])).to(DEVICE)

## 2. Data
Only `processed.zip` is read. The folds, hold-out and old answers below are stored in this notebook, so results stay comparable with the earlier runs. The suspect images are already left out of that list.

In [ ]:
EMBEDDED = {'dev': '...', 'hold': 'eNrNfVuPJjeS3bt/xTzOAtoBGRcG+WobWNuADQP+AQ1eNbK1o0GP1gv/ex9WV9dXVclKfZWd2V0zkFRdUsYJknE5wev45df+06+59F9/+vnzb//295/+8du/fa79p/Fr/v3Tv7rHf/rHf9J/+M//6z99chz0L//tf/7LTyXX3/vnX/Kvn37teXwqv/7y819//8n9hP/ojX/3jt8+QhnvQOmpUFH/8r///vMbH3rbwfprz7/+/tf/ty897jTEpzNbIk52sJjPxDJHO73G8SSs5W//9GdP/7TTUpGf/pb/tbcr8eWfdpov6XIFeK8DVK/H1z0Fgr9egbirwPUmILKngF3fA5J2FQiXK6C7Rhiv7wFNe14Y7XIFgt9TILnrFeBdBeh6BeKuAtcPgdmeEXp3vQZxdwy8vz4Uxf0+8PFyDZLb1YCuN8S0mw88X+6LtN8FoF+XK+D3NdDrNQi7Gsj1g+DdrjPK9X0wuemeBpcHJKKwq0G43BmJ9wOSXR4SSfe9MabrNdhlR/763Ey6PwrpcoZI++EgXT8IYTcrkLs8M5LtGiJ5f70GtK/B9Ykp7gZlup4dUNx1RqLr00LcrdaIr7eDJPsaXJ+Yku5rcLk38j5FIgnXa7BLD+h6esBuNy2QXu6NvD97RZqu12DfEq+fv2La98ZwvS/sUySyy2Mi74dEu9wMZD8aXD+DtT+FR9ezVN33g+spYth1A3aXG2EI+wpcHo9tt1pjf7kCaZcfMl0Rif5v/vWPFjH4khmTL8C6D6xXAdNuOcLirwLe72m5rKdld4KcNVwGTPvAdhWw7k6I8yVTHw/AYd+oI10GbPvAl7V4v7DlS5ZfvgDvd3W6zJ32w8eJ2fpz+D+ffv+cf/nb4y8+/fb5l58/ObFPsO2/fB5/8R38zCKpSoujeLViMigbeyethr3VceRUyP8+WhY26sHVMHqxknJs3Zn40XPIFj+GlkrWR+VcSisFnEtpaCgUXdSMvnUfQ8vYwtAuUoYwN1ccDVKn6NbhUxz1Y2iJDmvJam1d2fdoHV6hbCalV1E/PoaWrTgdKaZsOTBxVa8+OOaunWEN9kG0rBjskFrvEgvBbUL2sTbJnLu18UHscuSYeoZuLiTngs/kQkmwAVUuw+QOLR9/l/9ximrmPiFIf1UPf2zQp8EWa6wZ3Zly9ZL7kCAh7oagqDud+Ezp24/36YTKVnrjJg6KkXOlDq0lISKqL9XlH6FT0G48SvbdU25Dokh0UqEccxhcf4ROsbrkAoKJy+rMsmPLyqOFSI2C/BCdEiNEtAp/FB3UezPf8SM7L8FyCT9Cpxxh2gwVmmmVUbsrUsELejJYupcfoVMlmHIYcYTqAnlvtYERGmIac/W7ez7v1ukbwgJ8z3pXkCWS7n2omVIr1hEbykjkfkSXNSHkJBWZBGkosUNuasM7P8R81h+ik5JU2FcdEcoo9ZiHi9LUEL98bz9Cp1FiaD2MEAayYRdw4AE+VFvgnFP5XubOL9NM8jF0bjVzyxitNFKNUjWPaMWlHZ3UhWtz9UtNBdkawR38O7lGUBeZuwxo7QfGNZePo6lGAoMYCUMqkry6FJDJXWbE2+b6OFvTz7/9+98+/ePvv92pXkIGkqS1lgDdOgKI1sl3UTuUCvbzcToy9+hH0FJ7Kp1rHJ2SGyDlWTVWyR9H01FCMoqFEZJVfa9c2bcxu7qwRv6hmoansrYN7uR7GqBL6pOrUaP3EWXYpOZ+T0tv30tLcmSUiIovgRGKfODAiUPqRuL2R/37aTkt0FXk3zaQfpH2BJ0YuPZQxbGke7R8ctxzVQtxxFpKbDYUxQyjHEPig7eb9JQpfowOjN23kFB9jeByJPwVUVxoQOk1JwzKyVrekRifVEOGSSUUoyHcutTq4dPF12ClaXb9Y3RgSdQ8/tySjQqO01quoIAll5pltPEDOxDeAJ3ESs7DQ7HYihpldF+MJe8WZX+g2h+4zFa1z7/9nn/v7ZNz8ul//o9bjh4J9SGijHfRQICQX5BawDCiei60G7A1XTXEbyibcqbBrlomCdOPwf5n0c2F3Wh1fChlm4s9IhXmFuvwgSbRoFTV5kylidyj7EmB8abh06wPDet1yJDY0HEaQk3woZY6EnqMY1e7oNd35ZOigiqPqqZsPFx3FbVUA/cZ4iMPqfZhFA2oRb3FFjrlZAHVcym9I1eXYdlp/TCKRp+Kpa6lO3aJRFBBRNTT80cHFvxhFO2Ikr05s5RjGlAXHp9r9DqpRdkP7N9BUXLP3d2N4cacPPcEb4o1tTmPHiuJ5oQMv6dsTN9ZWc5QzjGBeLhOczFHqEpHSHBFJacPpSwYZomDEgrczj2jCFLkATN9YMXNfShlg6lWJWWE+Tm7EZRjrYEyS5tzMB9KWQSB6Cv1UbO16EvLPTXUQwiuCqv4WGaQhV3pIHu+VgmtOIoJeV/VopCT+KOVTU+RAJHKGrhJ755QCeVWhEAB3ZxsFr9nAoG+p6KWQUsr8miJNeeYkWiRBqgi/zOBCx5W9P1TvAvtIojeyDRrYHUlBtbQfEH9Gwrq9pTu0O4P16X2uPxCpdS6wZFrcC3YyK3BBuHpYHpUC2c6Q6XjHZY9XKMMh79qVKFcQJZbiIzww0n0ZLt7Z981xBomgw8jsFCL8IaMkhLhUmsBB/nBXsH8NDnEiVGlcaMAX5gTl9wJ9XipPaZ9whkkfvvs9EIlSqGH4kZF2EvOk0P/5RysjqDJ7a7F3K3S8Q5DUdOyh9H12GgWvNoMoU5Bhb3wj9aOiyBpBCbUt7mSFzAfb8OXPMhVyoe1O9vuoFXysw7rKHFS9U2aFB8kI94gytQPo2h2pqkIzNFsLlTWOeeXdCAcwkp3pwy+Knp6FLxpVxuqWGPrjSwhrbUQKFQvYt1JLuVHaHfbbCHBgV9RzhaMKqKy6TAkk1Q8mHf9QRuqFopaR3hx4IN1VO5zQ1AaHjQmeBL2IX8YRVNA/ynGNrnhPQraMQqFRCjAOLOOj6NomlQ1tJEdj06mPMIwB56IMsGq/zCK5oSkwqivEDVdTl24BFBDhM1MIrsT0t9X0eEcmde5qQo+34j84Dx3B8G7pNf+gxXVWw7KTjRpH/D6nAqJj2U6FkbeF7CiHUXtulWxlaItSsFfSJOxhBGyuZ7ndPqopbYRP4yiYeSMHnU1zKieskTC0HeuGpJQzx9G0ThMGtRE4UcRQ50jujMmFDUxQH/9wYq+XL2dzLePmKKPMcfpXXl4Si1T9Dn49mNWb99Qtvc+QqyIALn3FNIsKnqVmJFRqyT+0creVpsNFaugmk2oEbNHJOgOodXmJGzxdfyYVbSFosSEUtEFkJSUUPu47hOqbpc8MpRa+jCKJi8+114QTJFCLeSoIwZDhZQ7oar8OIp2qhTBTkdss8jtCKMh9IbKqMOzPs7QF6R5s1JThEdxmntbm3MIsew9HK1/GEUrHB4dOmN/DMWGdC9GJXpfks+x/GBFk3+aBayxR9cJZjl8B5HqnDsK9y4C9r87fWWJT5y+uqmEsXQdDj0QL6MfxBha8BAvDr2aOZ6h0t4mrIVKiWL2pVSey+MWUQPXHH0wszaaxntU+j7DmbmjRuOoNpC/a52nIXp3w8mYq5Hjwyg6/JwgjYyKDdUu+MZD9dadOkHttrsOdZ6iD1ecevc4V3ZbxY4Uvlxs+mxhe/XjowDvNwLYvUtA2ArQdwgg+OnDgbFnAoTfIYAdbQWEdwkIGwH6nj7grwu6zwXc2Qe3H//0Z+++HkV9Juh2Tej7RPFCVDgk6um2sGei7KCohVZmW1HPY+AbohZKPR3j/Fv+jF9/Wuu2+Zev5Iat4OQPtZYXw/l0AHLZxOfp5qUo24hK7phWogtRcVfU4yXSL+WobOX4eEglXbSOjrUu+IWoY8YatpaQ+KBWaSHqmGPHraTbjYHvk7ToKrFjohZddbuk512iEi1Epa3T3BEi0sKsjoVTcm4hKh0TtY2ByeiYqIULGh8Stej12+H090n6epHDc1HHep10ISocExUWog4ZO/E2gHrnDsqKC1n+WMeLW8k6Zg4SVrKOGbyu9KJ9i9hP0rSIp97JsW6zlXpyzCGNVrKOGb+thkAOJVhahHrv9JheSRaywjEzS6txDIdciVfDeCxE8yJEo9I61ET2K73iIRrBtBrGGI/JWplqTN/glbzqtWORn5+uPXsmC9Z/TBatZOkxWbySFY7JCitZxyx2kUa8P+RHi/IAFeCh3lolEX+MjOtKFNs3WGtY2L8/FqqDrESFXc76RqEXVu08FvTjwh39sSmAuAjT/hhnTatuX80m7PH7LzekLQKrj+9U6umSt60kPSIpLVz6vbMHX25DW7ghvTcAfpFkK0lHWid+JckOSFpRQvJHWrfwF6JDglYdzvuUcuPEX67eWnUTH+kmWwQWeqK5X5/2evXPFwLSSoDeG5keJsP7P37/9F//+798IvB/Jz598oIudk8rHpHayPhdS9GnNFB1F+2tSKvO5aRb/ECvT169E13paSre5hHE6MzFMZJaS1yGplhindtfv24Ef4G+uU/nDnQxo6f9ZbWpDzEWr6mGFvJcXQnqC0GBEm0Laf4ApErSpxW5PqJqlM4C1Na5B869xlxiGvZ1U/QLyPhOyIev58ePByw93S6C0lRaa6QeA6wjYfRb7+ypt2Bx0cVpM8Brn3kTXPyzja+hNUG3mmCcJcR5ULaXQNPN2W3A2fMO+OulsxW43vZTcxjSW4M1J65zF7C0ksEBI2WvcdvtjMjzbeDhaVeL02Dkc6w871nLpkNKbWV0qZxQaC7A02ngTC6loNqCG2mwxWLmGveofV6lJleAP60/O+qmHBxQPaeMFqv3vQbukqFj2IKz/0Zw09uRUmkmbgSNCWaWnHMljzEAX33zC4Nj+1bw+ATe5vE7H1tsDSENPu8lJdhc1e4TtQV4fCf4L//68yMj9jOE959ve/K5NC0aXMjNpyqJOmIptXkeMFjfYoudhc0lmMUBlCjOB0I0d01yLL1lTZ3OxfYvsEe24LOWwNrwtxZjHUN7L40qPS3mPsdWOoxN9uKaGYktzE396lOgqK5m0VYR8jxyXN6mTg7pMDTHW28XCzRyqJZDDmHejYBAkzMpa6eFidtxK+P0osXzQk7hyuQN9W/KDX9sPcHAkqQcF9DHW6zyYqCFU6JRqg2JVObVQ6PW6hP7lE10EdPirpG9vX3gpRoPUfWZn/VCocnotWE8BSHG95RTUfHdxbDo/STffEvBK41edUzIvSHPUaQ+r80McPpGc1MilwzStdBIT+mYl2q0uRGO1FlyVig7CsXFPkODkOVG30mNTrEa6CMhFprzyHfEMaTmYbpI/d9Njd4dTNUhFoJtWh3OqI7CBRw3+K9H3c9Q48tml1c5ycmnv//tSRkKocNHQkxgRIjRXSX6gtxMlWOybV4S9x4idIcGrE5qqb0Ngw51IEiDHViKDWkRsex6DUJRSl59DnDXuUc/ovBoCFpdc2KJ79Rgva9sX4M6ckde9AGZSgZjFIYWOE0IUOb9ffBNJoFs5UxyQy00KLPrHbwco4FI1uAm9foBATMMcJDRPQgpSCKqBj+QU9gmLcrftzuQSBA04KHSu3Swx4DUJTE1pNeBmvNtZc7csfYHKiJ4gElWbeKQd9M8ETOPHBhogFm53oBbYfyvzKqGWqlsIJdl7vkrvVca19jM7cxN7mOe/KtFe+whpdEHuiGYR9IZ5Lfo7yoll+i3KtpRggKhKg+PErMok5mlwq37TmWBzrTZDvrO3r+hc5k7/UIqycsYyO4JbtrxH5AfxWW5GL1VHWqae/eiI0fErJx6882huHTjTfSDh3zfVAQFRg4DATz2HsB0xwgC/7R5caKNxu9U5HiPmPnsOQdqqnnOWOWABOdBuGxE/Py2Isd98YYe07wM1yJFcH9DuSsa4RKSeo0k3K+1BgTFhPJWE1lIc9aumkMG0Q6XcDLateiVQ3y4KWReDC/UOxm4ZiqDa0UiKxejgzzR8OxCAT5GG+ShtS7Ina677r+bAY5GCLqiPppj7yhGZM7RM3G3XvZc8rgBkjwFw5ZcsDhTT2PNFa4HMoOyt+I/CrLl2BLDt4biG3qY2/Pn/Ap4g8QhOVEOxbj7VKivQvGZ6FGQ/UL00avNCzyN8XeXMRyuNF31/JnonYgzJKZI5hRFjQ9FDI6nIXUXLkG/zTy4hCxgbtQiSIeIxT5nqaxiPmoKWwIAGnUe+jxnU2sMvcUxKkl2Ml/bIKooyFMd16LXOa+I6no+6tGqG6Ep9YoCM6Z5AWm6Fr1LrD6RzLADGtAjaWcQgVCpZBV3MXoBv/LDpxYl1QLGhSK/+h7YuQxOci36yAVt1RznbVwl1ZibDAN2aC6a8BXotxUMHrNA5SAORZqr86pwCpS9Q6JvYTHJp5HOQ5eG3P6QbhDkc3AwuocLa9Qn2KLStegBJUYCGNxM51s2meD2OnN96dl1uxY9BnS2zeTifE0onFsrgRxxa3C7RcV+KnoCw0ARqODYGsu8KoHmXVrgXKPPs3XXotcmE4ng4OYS8nxWbnkofjPCqPVa9E46b2c0jgM1XNFWRargtxW0bxS5Fn1EhzqLptmP0pFdBUG3IM1GtB7/3RXoeuM2YJSlDkSW+VIIhrzPV19qKHPtDrxzgx7etYD0B+g1IbjEmubMvmWX5/XCPYHT5SQZtebF6HBt68Jgks4KeD7YFKrMKEJeW8rXovdEWhDfhXKeh76ddeqaEkcHrr0ocs9Fn+YNHkcqyahx8uj4gDDjW9TVquW56CjjEddoXuTZW4t1XiwMgmcmvepiH8QJ6OE2uWHFsjdKfYDWS0pxAjvpiqLSy5bbhKDnoVOYt8sjsJMMmveWwtEl8azvSynDXYvO1OczLjmWjJCTLOT5Top4j5JyPD1UdBW6cihZhnZE+WhaiFrOM8pnFHZe9Vr0ULMWA53n0s1Kaz524nlTPUdwzIt7PrFWP99tyAOhJlHlORkMlUD1umvxWvSW2hgzoRSf51W5GPTkwai6A99si20xp6L3yqAVHLNHWYG/EzPya9Uy2pzju8Tq7Hb9ZpilRJ33gITE3s3oynGu6IbJerb+bppOREcRhdCupCB0yZI5FBVgHBxyCQj116L7AnerNp8+nNsDYHi1N6RXFJeIudFfjN5KbtVCQxGfvWYHt+8UJ80ZMdnFPa8kHox+bgJDqLGslTtSDWiGI821X4s+U5ulPlc60W6igA6Yk3h9zis/vXVzGfqwkW2AyRBYPQ/YviY3pHNvSDntWnQEtVHjnL0RsJmEAgpOLiN6B2o5rva4GuYyOEoJlBBgOVpyzuzBL2YV3Wu+GN10Wl0YvpfIKCNRTHub9+c2T1HsTfT1lO1d6OlpfytYjJmvoDgzrejIMg+v1aCaZsLZZtjovnnG7IaO2iSJsxays7lyERyoJceUzMAwg1yLTvM6eBl19D4v55i1Y1cqiqgDjuncxeio3x1ohJaWSq5sPhv8PBqoNUradC36XKyTJhmJBo1PGIGCsh10w0bsfVFBn4peexdwq4Q046tHCTHanB/mQCYlLHahnIrefDXrc5PvJNU8CGNvad5lr8BfzNuci16LdUMVLSih5xX6sVdqyHDz5nwn19j87Y5YdLVDbh2B5gNqKbJUh3iP5C7zYp8tur8z1j3uz//Tnx/PB3zdt++j8h9s6b99+vVo4rOP4/0fy8uPyT0dN7vj4/TqY6/+7o+9vlbb7v+Y3OZju//jDfLTGaGvv3qWJxbfh83394+WvB7q5Nz9H8fNx/c3W183O3m9/+MNsr/fyMLr0bpd5XHHx7L5+H61w0Ztut+8bTNUfP84x9dGkuR+5LhR+x2OlTYfB7r3Y6T31x/bOz5Om4/vHir6en3Ps4/v7jDafBvt/m83H6e7PZLYb+LnOz7eBt+7DYy+IXDT6zhC7n5vptdxhNzTAd47Pn6dMhzf319h09n3OySFTZvlfmTbfHu/V7yOBPCx+4HTpsnh/s5OGxsJd9sIv6YV5OLdarPfdFjS+z/ejFS6u81PFyLcOMn9DsmvQwGKa7r/47BBvr/N/LrN/n6PZN7wMH//OAtv2nx/h8nrWODvjwUsmw6ju6P2a8u+nfz/4083FsJ399YmcN4O9f/ht0rH2XJ4PUg+3B35wqab7w8hYWNZ90eQTdwDJ7r3203k8vfHj7SxyfvDR7LNt3e64eNp/9f2Qd695/tN9Lodgb/v+9f6E70LnzcB8F5/fvw+HA0mj9/HowH48T6CbQC3d32/6T95F/7Gx0neZT/62u5J39V/utH/3omExysPNvan79LfNvrbu/rP0ub7d43fxnzsj9T//Os/HieA/sujoM+/1P6wp/afvf3zbR+D0gictRKNpiNyKague+6JnXucbb/BJv8TBL8JeptemLMoj/Hq2ZwDvIbvvZHhhSzvF7KeQtA7ZYWVLD0mK61kxUOyvs79vJBFx9pIq/56mpx4p6xVG+lYG3llE3ys70W2svQprbxP1kotScdE8ULW0w2h75QVV7LskKyw0iscc8cgK1nHut5WsuyYSZitZB0z1biSFY/ZRFqFnHRQ1sIdxR+yL1qFaKFwTNaJeq38Ufyh/vpa+r5sIh8Txed1F69kse1emf1sheWlrFV38bHuWkVVkYOyVnqpOybLVrKOmdcqrMofhK+3+j74laxDIYfCIkyIHbPVVXeZHRO16q54zOyXouK9F8W/FLWy1HSst9JiFNX5Y7JsJetQ17M7j+PwiqvqMR7Hq0iozMdkxZWsY3rxahyFjsmilaxjbVwNo/pjolZN1INqrbo+HArQrG4l65heKw8KhyihLEKqHmOXy846FlKXfRXdkQS0Kjc0HrL4sOr2dKiBYRVrjhHeRa8Hf8h1oluJOtRVaREbAh0yq6QrUQfC3+NM7UozDofF8UqcHRa3auwRpvo4LbwaUdHD4hYOEI5wzMdJX78Sd7jvZDWy4bi41ciGw3a3io7hCHd9FJdW4uSouJXZ2WE70dXA2uGRsNVIpMNOsaLr5g53XVz0nR0JwF/EpZW091/T+x8fJM6XMJ6dVmhDHeuIwn2+2CwcY8gx1h4q0QI2vPPm2EdQT08XeMY2L5PsLsZ55Fdt3pibWiga+nyw0W9B2R8EvZ3ybpEz+ZZ6p5wZdpx6Rut9mZcQj9oWoHwIlFhvVwEPK8Vr7lnrCCCcTZIGy20ern88ffUCU97bu1+M5PEaB/f8NiWXNPa5K5m46UjVtZKHiGvitHguW3D9JvDwEtzQUtJho86zAFE1uY4hT6ba29iCg0AfB1f/4lli33U4y+pHys6cL4NSbNXBA3umBbjxN4DfTl3lipBhsXpiEYy4UEHJn9lUQh9mW+DoTwFu0Vm1IjkHxyzGueZULRcnzY3Hl07PAzb34tJO1J1aCJi9Fe+4I5rUeYieA0mjhWdFOw3cEiyvZeZUhuUAK6/DCSpOy/PUp10K3p0MKb3FXOcpZkrqUbzNq/d6jRrTaeBPj4y+vDDNMYybQ6eYnc7noSmxq6Jxnmj2j/elvMB/vATyJPzR+7w/oRoCTW2x5WStlEB5hBqk5fPxX0aZkSGQMfYSVQSRtvnRM5yBQ2hwvNOizBM+v3j/HJ0fcyKPXGlRmZpGRLpcKbgoo21DbCR/Jj7V0RvRPF2YQZJDhyPMG6N6SpHd480l5+K/uK63ZCqWJcEOBA5goSDP1VBqrYm/XlP2Ep/PxJ83S7IFDHsAcRH0Q6rdYPrFhEsI5+Pr7XHpEb0HUUrzeN2QXJtz3hyCcFCW3E9LMgvs+QSyw5CzxnkNNTPCrSdEAw6xWbgUOwfyvbqgBUVeBUmat6iAUiEIu3mHygXY8fmYg9VQQ6zLwah7NUL7NQ/j1itZW9i8yLfh35526OBNfnBuwqmZZ+d4lEQS52U9PLb9HvXdt6J95ej6xCK1OZPGKRe1ymNESsGLleHcvO3WbVHf9brDG6ipBk01lQxaKcmlYU0bGoTAHhDnx6mo4XaLbgcp76gDukmed+ANzUxxiIWW2sKjo3vHayFvoFohJG0LoA8I51rnXFl285oGB5Ll3D2of3Si7Cvqs1sxqCTt8x0HkJccmb01mNM8xTxro7ZAPdzDz+6jCOxDIWQK71oYY16KwVnn3sB541o+FZVvN2viT8hRPnd1BVQp1owsbcidqEx0W2fG5A6jPtkwCq0aMI6qoyNLZXEobDuytKR54Y8tUOko6u2VBuQBRKbESDYBVW8OzFk8XLap685tUZPfs+Fda7qhgvsNk44Y4dWFMl/d6dbmNWIR5bWMN1EPtDXdHrtxfQZem1f0qwWMK/5fYuSHw8A1LFDTQVRUFk8RsTiPWAxrCuJHTtFKRi7w87x3qrzl3YlOQM0DmV7V1Yc7BlwJaKEOWDWqfHjVOBPV3+6LArWMIJSIEmD2qXc/r0MF10FOLqg9tqjivh1VK0yJ51sXFhszLEmNSy3kIlvK5RrUUAvbvAqtNyseSEjviIeOS7NKEk5FvfkrRi+gcGADd0JokIHi2ZV5e0vMseoCNX07qoG7zkLJ5m0wSqWhhox93uAPOlm1XIOKAgD2GxAUzaNsHeZjw1B3RoaFQqei3l7hYp7zbKkUBGIrrppHpGjsCaQZjGZhw8qHUZ/aqrHRUEUp4i0jOIWYUCWCoFeUgcNtq+MU/LejogiuMdsYoOSucEpaYb0JH5uJT3omqjzdgcFgDFrmFT+hpKFIQcXbnA+Y5ztaXLQ1HkbV25XpCg6j2Ym25sWj3YIQOee9krRKW96U0gmoLZBlFPzIbOj34sJA8OdS4kA1SFzPRI1PM+WFH258YKRWhME552Fztq9yR9UR3abuQHHgD2Z1/+yug0mcZN4FPKiqM5e0KAJGxn9Tsx8L1KNs7Rnq8FFinI8msICshdFhR4yRzSIjdT0RFez6KSL2oK4B2qj5yi6lAabhu9ZoIQttUbcX8L8f9YEzKAof9XW0Do8BkSLfWpGIqOHeRD3S1qcoESKSOSiEL1Z7RtkeKmdLBZFEqqS6QI3vvWb8K6o+sbUSUE0pWOFMMKBpvaOUttTizHijhC0qhaNtvb3XluabeNlchEcM9Dw4UxlpNr3X+bLTFpX1BFTQ/NAl+9BRsnr0cnnIACnDtsd2VvobUNn9Mz2dDREC5Z4vWYCPxooiuglyQTdUHAlEbtHawxzmBa6W2MQ3h3Y0JXEoYmNOrdtosCjXT8TV9NTLQ626kBpqm+AnDddAvc25sfpwrewCNR5Gvd0zNjn/fCYro6wT0+zbw70zAb/2TIu26rszwPNZ/1ez7r5xRc2cM0IG8mwz0JoYYgQhnzPwW3Sj89DJEcgpquaMDkFP9zhTPJKwztdlcrsWXcxqiKgMYGdxoLdH6TzvlG01oZPtWvQK/50zbmCRKHGbIgtTy64RYgnSZLgWffh5Weu8Z6x6KuCQ+J8gH8+nCRvK/IvRI/JgbS3N5/ECLCCTzsmN4Guwcf6439i7y/MFPCcZ5EMHD59lvrVZ5koXYviWg3iVc5BTRcyuudaC3EEIqbFgBEordc5zj3AdcpH5cJlE10ud98XC5gtKRJhaYvRBvQ65ZmNQaEaSasoAH61kC6mPBF69nRU8D7lTLh5Mer5gq6CY3Qr6IHBOzCOyXIgcDdGLDbHM5XmRoWFw5x0moXbO7cJxHlF8RxAZCh4IChZ8nisLPaIcB/3spyO/jKZ1Xo47V8+0FHT+3LQwL4mFwZcYKG/RQzwPPZroAOFGDWceCawM58HQIul8CYLtWnSQlSypW8ww7FgR1sfwFuqcOOxhNeanorcEypDnnLCWOdPVEoEqenQIaq1GZ6O/WrcF/0Te0jSvRx1gqfD3NHzzVuHxtJm/ZHL+PHRBYcM9zDnEMNzQin733s3oxiUZXYsO8lg6dV8iFbAnAe0Oqi4GhPkZ4K9FT86HzPC11HyfT73Wkv1DJYZoZ9v56pPR2eZ2kSw2l0zFXJWQXU8i3TuQ52vRC+I7ygI/n+ekEed11LBC8GbQCXFNr0VvPFBbhw7S0G3evF/h7wQ/YLFAwZ+O/pI1d3FZeGYyFCyuOmnNYoTD4bdCK6v7Jn+XFxvS/FxyczA4KaMgsYT5uNKcFywVnLlv5xjmRROnoauCnII7JqFkkykL8VDXR/DDB4pXoyOkEso187m5WFAUDxGUL/NlUmTda9Gt+LnXtEjJiG+RGB1OfVaOuXpXL2671drBkYvNV8tzS5lTbwPF8ZgvYXS6GL310qpDZosh1RgdeKube1NIJMiQa9EjagMuFVX5XLJq87niEZBkZT6sZXTxuKcgiC2UbShaXMDwEGkkW4uRe60Xt727DEeX1oeUljxqpjRvw0cxAXbJ43R/1/Ay2sznHTqbOQaf6oG5IevM7a8FlfO259nLieiALTmOZODQ2gEKK0CvW21gWy5djI4+R+kiTv18YaVZYUtzs4wyfl7Y/KnoiDJR3ShEXRF4xJdaUk8wP7Cd4fVa9JDyaKId+RRGjwwTAspW6+pAdvKimjgVPZWHJz0SQPNcLigdiWamN53nUh2djW4vMyyiacpx3j+PgorwJ44xR4+qORFsb4tu4Tx0BqOpuepA+Qyrq3FuvcD/Bo0EB7wYXebGwxQezhVoiLOMgp/PtSq2ubB8LXpQcGrQibnpvEuGzSfw6syOmlXf+sXoaDbPjbciiPaxzXdmVFoSpFl1FK5FtzZqY9a5p0h66CXMN5yJqXqq2i/u+RISp6h1vh5Ic+tpmg9HPrwY28C5Lra6YhpTgX+P0RFcKyqp1l2IweZOcJVr0ZHW00CLQ1WU0Q5GP0IYhCA/I06h09GfVjy8H/N99OFBq+bt/8yuh5LSqJHBqxb2nvQcZMrDJHZTU5U5D+7my1bsUiUDw6frkLl7ibn5ikQOa5vnG7LGMF+J9kh3F7ZZkhtRk/UanZvr3/NliZwz+TjQFf46ZJ2DiQoF4dzKbPzwzuesFSSyhe3e/hORow7Q5gaSDvbIsHSCUc+XykKbLwtdh2zZt/noMkJKgTd7Q6mQJftSCrmQLxznOhegYwdfDyMr+tcE9WKhHKBG2p4XOw95zImBHpA9qKI8RAeArXX80UVf5Mw23w7RPG0pdNE11ISWMbYSLacemXv2IPAPC5wbcOF4GrgHV7HZWmSLLDCtAePK3hOqFPL+WnCE72bzUTaa+ykNWVNboc51jj0qtmvBkT5QjXUwBt8q6KlAmVpDnFMEI/pLwUNM7HMdvQZkUAN7Uas95uIBH0wvBY/KeQTXxSOkwe5MWAfSyHyuiHuQK8BfTcR5RBcEGZQF3cCPuUpsDiV67nMD/HnLmG8oEEAM5ztdrgQOyCLgq+zDQApnUImeL1dgUvXunQ5kMvE+CyIPWBtUSQ0ERi5XIGkCaQ2giV1HnY9xdp1Pj3OIKq7a9QrECvbocy9xzod5NpSOknqmuXtkUaydrUAha7mVoEwoFyUW5B7XKlIPArA6d70CoaiWnsAj8jwbLYj2YlDFNZB6vt4Ia699lKqpAttlpD2ZC3CNpXnkwnC5Ao3A2UsIKaO/FYVqI1SP8+QeikcQ3csVgOH1XPvDQYk4CeUAtQvZu7k4Fn28QAGvtycq50lp6mFoqR4DHxnFVKzV5jvk2xPqLOn9D+a9Ojn7ovW+zpeuFSXF4OaqcA0999HCSB7ZKZ23/vmGAlRcKQi3iD/zlD7B7h529yiSIQbBXa9AQ8xBCsQwRJhemud0XOzVeO7hinS5AkIEvp2LGxn126wytCAgsTWPPFDy5QoYe9bae0eST4aOqAgCMSAjeQEx1csVQPhrpYJ9z73yKrUnVwhFpuOBnuF4gQL84oqMXJBzQLglOhmkg8HGxnwcPIpa1K0RqqZTFSgREW+uO3cycnPjPjJCy2DkZe7ODZcr0KDAfDIUOWAQkbjEAzXAPPzZvTJdocDLI/SqCATOfDIhZGRpUdKcN0c/zB3C5y0Lv6VAkeG4JrKCEjeg9kjah59vmTrT1QrhyQoQCkCuriD8jsoW2Hp3Y3QidA2NerkCoAE22Q9c/mGaBbWAcXlYI0+liV2uQCMUgqVVzmBlCbUR8i+Bm890UECMrlcgWlIpURMiYbWI6gxpSa0MztV9ByOEyyP3BD+3weQkQWpBkQJ2oJFCHFcMwcsVS5epgxJ6aaBixZOmEKTPHSqK6BhPXKl+S4FSRedi5TwFkUbSEUdrhHg4n/gL1yvAviD2ejCh2OYGwCrQAsUZIxAFX/z1CqTgLaXuy3Dz1EKb74yO+TCCtUKLza5nKxAj8sBcPUJVMHrJbeTWqVKGT7TV6uW3K/B64RqtRgR0qIzARjLUQaGW5jaG4WKJ5y0hvqVAbu5hNyB4eEJObooytYMQdT9EyrhcAZqLVpoMBZEEP5fwQcsSrEB05LhazjpZAZZaGQUpKLhYZhEEBpmPLzafVfv1CkiPkQm2J4h9yM08H9sFG6IWw7BSL1dAeW5NHJMPxvmXAxdEQVqMnbcY8/UKIP/HeZmRhTov+4AJwAFjCpRTAWO/XoG5iaHO0/TBayqIg2DFcy0C/eBTt8sVmIcPSxwRXjDLsqDzhEsKWnof5hbbZ74qUHL9vX/+Jf/66fHf/vLzX39/z2/v1bAGyaWUgbIdRClGo9ZrAGnLA+FC++VdhAIpxnntGaE/UDl1ZAnlHKLOubTsrlDAXuRryYzqPTDKNY+gESaJjYLE6SSU7dVnHCyeq0CKbaRQGjOjhgSL9x61s6JqQRJJ7nIFUCa1uc0MRN0XcHiax5+5MbWUlhuZz1YAzsFSEgo3peTasIgK3hCn+kAZvSifz1YAKdKh411H8TC38yJIz3MD5nv2XhdHos9WIHvUbjnQvAYLbKE+XAKFgOUZTMqSv14BaYFD5TbnblOKYESgsNMz54UDXS5XYMDeRitIkGi0DOlD5mLSPIs5twBdYQNBX7ghDxRM0vLcaZcftjxWFC6xwRQDLYzQHJ+qABlFVEg8V9OY0HgkK3EVRVQtdSwmEs9WANVyRsjjkMbc8me5K8KyH6DSYG4i9yiwTlHv1yXYfIyujdKJVT2KuvlEcOPsPDrJ58s7I7vIITjjeUYQodlzT3keVsQoNZR0lytQPCLRfFS58bxaZC4rEPJ0GNXbWFxGeoIC5m+3KRdDHixSIuqXkZhCGC0ZwUm7LPY8H96VtAC3uU2jo3bVgrToMhzRk5tctveWz9yMtgCPTrXFPk8k++ashRh41hC5IjTX5K8GzwFmlkrMFsDcUyuwAiSDeWUeu2vBda7dV5oTWE6nCvg8eqGCH13pl4LnSuwRdkZFCEwoGsQJze2gsECE43Ip+LzAR1AwO0JkAfGlohVmF2WeaCuLW0JOBR8VbJcR4zyBAHEvzbc6b3yAxcm4FnzeelCQ3GDeyHWgQB1uDjKELBRDXizd3A3+8OM/av61/enPz+/Lf/gVgtXLy/y//Hb540tJzq1khWOy4kJWPKaXX+n18mWr+2XRibJ0ISu5Y7JsJYsOySLZyorumF6kK1n+kCxetDF6PiRr1UR/zFR1pRYfUyu4lSw7JmvhQvOa2iOyLC1k6bFhXKmlxyw1rtQKx9RKvJKlx2St9LJDHkSrqBqNj8laeaMdil60HMdj0YuWUSId04sW7pjcQb3SStYh+3r54uZXWf6Qa6+6KxEdE7XqLkqHZIVFckysx2TxSlY4JktWso51fVj1PR8zVVv117EI/fIl0CdZx8ZxFVbTsbD68vnOJ1nH2phWbTwWVlfWZYc8iP1KVrRjslbemA71PK+MK8kxUUtZx5q4YpcpHTJU3va8OMfHRPmVLD0mKy5k+WPWJStRx0ZRaCXrUEhdtpAO9ZasOp4OBVSxhahjTFziaaI0LUTJIVFhZfByyA9fPGj9VZQeiqVh1VfHGL2tGqiHGhjdQtSxxBNXdmWHonJa+bMdamBadXs8MIKPT2Kuuv5IvngUt+qzIylj8YblozjvwlFxK5/0/rC4sJJ2uK0LGi2eDg/syhH8kVi9fSXyqzS+W7m/5c+ff/v3Tw9nCeY03yObfv7rL6cNIPRrDtj+yzt+s0Gyt5HEnYoUd5D8qUjpTSTRdCaSf3uciM5FertNPvKZSPQ2kJ4LJDtIpxo57XReOLVNvAMkpwLR97IHke/lTfLdbFzD20jp1FgU/A6SnYr0dpvI0alIcQfpVCu3nfjqTx2nuNOmJ1JwClLy3ytnpLdjBPGZuX0P58xATu7t+EpyZoSgnUHSU7tuJ2WQnulMRDtIQU9F2glF4VSDINtBOtOZiHcCRDzVImTHneKp47STcCmdmTJoJ+FSOnWc9O1xYndmytgxcnanxqKvq99LpFNDROAdJDsVaWeYTs3stMMh+Glq4xSknSbRqU2KfgfpVNOLsoN0auelHXfiUyNEsu+EhHjzNpLoqUhvR3I+tWp/WopbIYUzcwbTjuOaOxVpx3Pt1N7jHYuwM/2Jdxz31FqaZadJp9IV1p08mM7Mg/K25Yk7s0k7BEzcmaFop0H+TFfaIRByahkddlpEZzKVt8MdMuCZODvTbHxmZLAdmzt1Hj7SDtCZ7hp3xkjO9KIdjnLmVOiXBaWdiQEJ4WywtzOgnJhrv4DtTESInd0y3hmzePaY7fhw4rOxdhqW4slg+na41RMLw5cLvSswf3Y37mQs9XI2mO2AnT1mO4Fe+Wyf3gn2ymd3Y9wZM/42lvH/AQFiMJo='}

In [ ]:
dev_df, hold_df = [pd.read_csv(io.BytesIO(zlib.decompress(base64.b64decode(EMBEDDED[n])))) for n in ('dev', 'hold')]
assert (dev_df.flat != dev_df.label).sum() == 174 and (dev_df.hier != dev_df.label).sum() == 159, 'stored answers are damaged'
assert [int((hold_df[f'flat_m{k}'] != hold_df.label).sum()) for k in range(K)] == [37, 38, 62], 'stored answers are damaged'

def find_kaggle_data():
    # Kaggle may unpack an uploaded zip, so look for the class folders first and then for the zip itself.
    img_dir, zip_path = None, None
    for root, dirs, names in os.walk(KAGGLE_INPUT, followlinks=True):
        if img_dir is None and set(CLASSES) <= set(dirs):
            img_dir = root
        if zip_path is None and 'processed.zip' in names:
            zip_path = os.path.join(root, 'processed.zip')
    return img_dir, zip_path

IMG_DIR = None
if IN_KAGGLE:
    IMG_DIR, CFG['zip_path'] = find_kaggle_data()
    assert IMG_DIR or CFG['zip_path'], 'Dataset not found. Upload processed.zip as a Kaggle dataset and add it with Add Input.'
if IMG_DIR is None:
    IMG_DIR = os.path.join(CFG['data_root'], 'processed', 'images')
    if not os.path.isdir(IMG_DIR):
        assert os.path.exists(CFG['zip_path']), f"Missing {CFG['zip_path']}. Put processed.zip in MyDrive/rldd."
        with zipfile.ZipFile(CFG['zip_path']) as z: z.extractall(CFG['data_root'])
print('images read from', IMG_DIR)
json.dump(CFG, open(f'{OUT}/fresh_config.json', 'w'), indent=1)

if SMOKE:
    dev_df = dev_df.groupby(['fold', 'label'], group_keys=False).sample(n=14, random_state=0).sort_values('fold', kind='stable').reset_index(drop=True)
    hold_df = hold_df.groupby('label').head(10).reset_index(drop=True)
for d in (dev_df, hold_df):
    d['path'] = [os.path.join(IMG_DIR, l, f) for l, f in zip(d.label, d.file)]
    d['y'] = d.label.map(CLASSES.index)
missing = pd.concat([dev_df, hold_df]).path.map(os.path.exists).eq(False).sum()
assert missing == 0, f'{missing} images are missing from processed.zip'

FOLD, Y, GROUPS = dev_df.fold.values, dev_df.y.values, dev_df.group.values
Y_H, GROUPS_H = hold_df.y.values, hold_df.group.values
IDX = {k: np.where(FOLD == k)[0] for k in range(K)}

def split_trainval(frame, seed):
    # A grouped fifth validates, the rest trains: the same recipe as every earlier run.
    sg = StratifiedGroupKFold(n_splits=CFG['val_splits'], shuffle=True, random_state=seed)
    tr, va = next(sg.split(frame, frame.label, frame.group))
    a, b = frame.iloc[tr], frame.iloc[va]
    assert not set(a.group) & set(b.group), 'group crosses train and validation'
    return a, b

def fold_frames(k):
    return split_trainval(dev_df[dev_df.fold != k], CFG['split_seed'] + k)

VAL_SIZES = [len(fold_frames(k)[1]) for k in range(K)]
if VAL_SIZES != CFG['expected_val_sizes']:
    print('NOTE: validation sizes', VAL_SIZES, 'differ from the earlier runs', CFG['expected_val_sizes'], '- test folds are unchanged, but report this.')

# new-source test: train on `named`, test on `r6k`, development images only
NAMED, R6K = dev_df[dev_df.source == 'named'], dev_df[dev_df.source == 'r6k']
overlap = set(NAMED.group) & set(R6K.group)
if overlap:      # the few source photos that appear in both are left out of the test
    R6K = R6K[~R6K.group.isin(overlap)]
SRC_TRAIN, SRC_VAL = split_trainval(NAMED, CFG['split_seed'] + 100)
print(f'{len(dev_df)} development, {len(hold_df)} hold-out; new-source test: {len(SRC_TRAIN)} train, {len(R6K)} test, {len(overlap)} shared groups removed')

## 3. Toolkit

In [ ]:
MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
S = CFG['img_size']
train_tf = T.Compose([T.RandomResizedCrop(S, scale=(0.7, 1.0)), T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
                      T.ColorJitter(0.2, 0.2, 0.2, 0.02), T.ToTensor(), T.Normalize(MEAN, STD)])
eval_tf = T.Compose([T.Resize((S, S)), T.ToTensor(), T.Normalize(MEAN, STD)])
AMP = DEVICE == 'cuda'
IMG_CACHE = {p: Image.open(p).convert('RGB') for p in pd.concat([dev_df, hold_df]).path}

class LeafDS(Dataset):
    def __init__(self, frame, y, tf):
        self.paths, self.y, self.tf = frame.path.tolist(), list(y), tf
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        im = IMG_CACHE[p] if p in IMG_CACHE else Image.open(p).convert('RGB')
        return self.tf(im), self.y[i]

def loader(frame, bs=None):
    return DataLoader(LeafDS(frame, [0] * len(frame), eval_tf), batch_size=bs or CFG['batch_size'] * 2, shuffle=False, num_workers=CFG['num_workers'], pin_memory=True)

def dino_features(model, x):
    t = model.forward_features(x)
    return torch.cat([t[:, 0], t[:, model.num_prefix_tokens:].mean(1)], 1)

class DinoNet(nn.Module):
    def __init__(self, n):
        super().__init__()
        self.backbone = make_dino()
        for p in self.backbone.parameters(): p.requires_grad = False
        for b in list(self.backbone.blocks[-CFG['ft_blocks']:]) + [self.backbone.norm]:
            for p in b.parameters(): p.requires_grad = True
        self.head = nn.Linear(self.backbone.embed_dim * 2, n)
    def train(self, mode=True):
        super().train(mode)
        for b in self.backbone.blocks[:-CFG['ft_blocks']]: b.eval()
        return self
    def forward(self, x):
        return self.head(dino_features(self.backbone, x))

@torch.no_grad()
def predict_probs(model, frame):
    model.eval()
    out = []
    for x, _ in loader(frame):
        with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
            out.append(F.softmax(model(x.to(DEVICE, non_blocking=True)).float(), 1).cpu())
    return torch.cat(out).numpy()

def class_weights(y):
    c = np.bincount(y, minlength=NC)
    return torch.tensor(c.sum() / (NC * np.maximum(c, 1)), dtype=torch.float32, device=DEVICE)

def train_net(tag, model, tr, va, param_groups, epochs, patience, seed):
    seed_everything(seed)
    w = class_weights(tr.y.values)
    dl = DataLoader(LeafDS(tr, tr.y.values, train_tf), batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'], pin_memory=True, drop_last=len(tr) > CFG['batch_size'])
    opt = torch.optim.AdamW(param_groups, weight_decay=CFG['weight_decay'])
    steps = epochs * len(dl)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=[g['lr'] for g in param_groups], total_steps=steps, pct_start=max(0.1, 2 / steps))
    scaler = torch.amp.GradScaler('cuda', enabled=AMP)
    best, best_state, bad, hist, t0 = None, None, 0, [], time.time()
    for ep in range(1, epochs + 1):
        model.train(); tot, cnt = 0.0, 0
        for x, y in dl:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
                l = F.cross_entropy(model(x), y, weight=w)
            opt.zero_grad(set_to_none=True)
            scaler.scale(l).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += l.item() * len(y); cnt += len(y)
        pv = predict_probs(model, va)
        f1 = f1_score(va.y.values, pv.argmax(1), average='macro')
        hist.append(dict(model=tag, epoch=ep, train_loss=tot / max(cnt, 1), val_acc=float((pv.argmax(1) == va.y.values).mean()), val_f1=f1))
        if best is None or f1 > best:
            best, best_state, bad = f1, copy.deepcopy(model.state_dict()), 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state)
    print(f'[{tag}] {len(hist)} epochs, {(time.time() - t0) / 60:.1f} min, best val F1 {best:.4f}')
    return model, hist, time.time() - t0

def save_preds(path, **arrays):
    tmp = path + '.tmp'
    with open(tmp, 'wb') as f: np.savez_compressed(f, **arrays)
    os.replace(tmp, path)

def release(model):
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()

## 4. Frozen DINOv2 with a linear classifier
Features are computed once for each image, plus three flipped copies for training only. Test images use one view, as the saved flat model did. The classifier's weight decay is chosen on validation images.

In [ ]:
FEAT_PATH = f'{PRED_DIR}/features.npz'
ALL_DEV_HOLD = pd.concat([dev_df[['path']], hold_df[['path']]], ignore_index=True)
if os.path.exists(FEAT_PATH):
    F_VIEWS = np.load(FEAT_PATH)['views']
else:
    t0 = time.time()
    m = make_dino().eval()
    @torch.no_grad()
    def extract(frame, flips):
        out = []
        for x, _ in loader(frame, bs=128):
            x = x.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE, dtype=torch.float16, enabled=AMP):
                out.append(torch.stack([dino_features(m, torch.flip(x, v) if v else x).float().cpu() for v in flips]))
        return torch.cat(out, 1).numpy().astype(np.float32)
    F_VIEWS = extract(ALL_DEV_HOLD, [[], [3], [2], [2, 3]])
    save_preds(FEAT_PATH, views=F_VIEWS)
    release(m)
    print(f'features extracted in {(time.time() - t0) / 60:.1f} min')
N_DEV = len(dev_df)
assert F_VIEWS.shape[1] == N_DEV + len(hold_df), 'saved features do not match the images: delete preds/features.npz'
DEV_POS = pd.Series(np.arange(N_DEV), index=dev_df.index)

def fit_linear(tr_idx, va_idx, seed):
    # tr_idx and va_idx are positions in dev_df. Returns a function from features to probabilities.
    Xtr = torch.tensor(F_VIEWS[:, tr_idx].reshape(-1, F_VIEWS.shape[2]), device=DEVICE)
    ytr = torch.tensor(np.tile(Y[tr_idx], F_VIEWS.shape[0]), device=DEVICE)
    mu, sd = Xtr.mean(0, keepdim=True), Xtr.std(0, keepdim=True).clamp_min(1e-6)
    Xtr = (Xtr - mu) / sd
    Xva = (torch.tensor(F_VIEWS[0, va_idx], device=DEVICE) - mu) / sd
    w = class_weights(Y[tr_idx])
    best = None
    for wd in CFG['linear_weight_decays']:
        seed_everything(seed)
        clf = nn.Linear(Xtr.shape[1], NC).to(DEVICE)
        opt = torch.optim.AdamW(clf.parameters(), lr=CFG['linear_lr'], weight_decay=wd)
        for _ in range(CFG['linear_steps']):
            opt.zero_grad(); F.cross_entropy(clf(Xtr), ytr, weight=w).backward(); opt.step()
        with torch.no_grad():
            f1 = f1_score(Y[va_idx], clf(Xva).argmax(1).cpu().numpy(), average='macro')
        if best is None or f1 > best[0]:
            best = (f1, wd, copy.deepcopy(clf))
    clf = best[2].eval()
    @torch.no_grad()
    def predict(X):
        return F.softmax(clf((torch.tensor(X, device=DEVICE) - mu) / sd), 1).cpu().numpy()
    return predict, dict(val_f1=best[0], weight_decay=best[1])

pos = lambda frame: DEV_POS.loc[frame.index].values
LIN_OOF, LIN_H, LIN_INFO = np.zeros((N_DEV, NC)), [], []
F_HOLD = F_VIEWS[0, N_DEV:]
for k in range(K):
    tr, va = fold_frames(k)
    pred, info = fit_linear(pos(tr), pos(va), 100 + k)
    LIN_OOF[IDX[k]] = pred(F_VIEWS[0, IDX[k]])
    LIN_H.append(pred(F_HOLD)); LIN_INFO.append(dict(fold=k, **info))
LIN_H = np.stack(LIN_H)
display(pd.DataFrame(LIN_INFO).round(4))
print('frozen development accuracy', round(float((LIN_OOF.argmax(1) == Y).mean()), 4))

## 5. Fine-tuned DINOv2
Only the last two transformer blocks and the classifier are trained, one model per fold.

In [ ]:
HISTORIES = []

def make_finetune():
    m = DinoNet(NC).to(DEVICE)
    groups = [dict(params=[p for p in m.backbone.parameters() if p.requires_grad], lr=CFG['ft_lr_backbone']), dict(params=m.head.parameters(), lr=CFG['ft_lr_head'])]
    return m, groups

def run_ft(k):
    path = f'{PRED_DIR}/ft_f{k}.npz'
    if os.path.exists(path):
        return dict(np.load(path))
    tr, va = fold_frames(k)
    seed_everything(200 + k)
    m, g = make_finetune()
    m, hist, t = train_net(f'ft_f{k}', m, tr, va, g, CFG['ft_epochs'], CFG['ft_patience'], 200 + k)
    res = dict(test=predict_probs(m, dev_df.iloc[IDX[k]]), hold=predict_probs(m, hold_df), train_time=t, hist=json.dumps(hist))
    save_preds(path, **res)
    release(m)
    return {a: np.asarray(b) for a, b in res.items()}

FT = [run_ft(k) for k in range(K)]
FT_OOF = np.zeros((N_DEV, NC))
for k, r in enumerate(FT): FT_OOF[IDX[k]] = r['test']
FT_H = np.stack([r['hold'] for r in FT])
for r in FT: HISTORIES += json.loads(str(r['hist']))
print('fine-tuned development accuracy', round(float((FT_OOF.argmax(1) == Y).mean()), 4))

## 6. New source of close-ups
All three models train on `named` images and are tested on `r6k`, which they never saw. Only bacterial leaf blight, brown spot and leaf blast exist in `r6k`, and the same grouped split is used for all.

In [ ]:
def run_src(tag, build, epochs, patience, seed):
    path = f'{PRED_DIR}/src_{tag}.npz'
    if os.path.exists(path):
        return dict(np.load(path))
    seed_everything(seed)
    m, g = build()
    m, hist, t = train_net(f'src_{tag}', m, SRC_TRAIN, SRC_VAL, g, epochs, patience, seed)
    res = dict(test=predict_probs(m, R6K), train_time=t, hist=json.dumps(hist))
    save_preds(path, **res)
    release(m)
    return {a: np.asarray(b) for a, b in res.items()}

def build_resnext():
    m = timm.create_model(CFG['resnext'], pretrained=CFG['pretrained'], num_classes=NC, pretrained_cfg_overlay=overlay(CFG['resnext'])).to(DEVICE)
    return m, [dict(params=m.parameters(), lr=CFG['resnext_lr'])]

SRC_RX = run_src('resnext', build_resnext, CFG['resnext_epochs'], CFG['resnext_patience'], 300)
SRC_FT = run_src('ft', make_finetune, CFG['ft_epochs'], CFG['ft_patience'], 301)
for r in (SRC_RX, SRC_FT): HISTORIES += json.loads(str(r['hist']))
_pred, _info = fit_linear(pos(SRC_TRAIN), pos(SRC_VAL), 302)
SRC_LIN = _pred(F_VIEWS[0, pos(R6K)])
Y_R, GROUPS_R = R6K.y.values, R6K.group.values

## 7. Results
**Decision rules, fixed before any result.** Each version is judged against the flat ResNeXt-50.

**Better accuracy** needs all four:
1. Fewer development errors, with Holm-adjusted source-photo p < 0.05 across the 2 versions
2. No fold worse
3. Fewer hold-out errors, averaged over the 3 models
4. No more diseased images called healthy on development images

**Better on a new source of close-ups** needs higher `r6k` accuracy than the ResNeXt-50 trained the same way, with Holm p < 0.05.

In [ ]:
OLD_H = np.stack([hold_df[f'flat_m{k}'].map(CLASSES.index).values for k in range(K)])
OLD_D = dev_df.flat.map(CLASSES.index).values
ARMS = {'DINOv2 frozen': (LIN_OOF, LIN_H, SRC_LIN), 'DINOv2 fine-tuned': (FT_OOF, FT_H, SRC_FT['test'])}

def macro_f1(y, p):
    return f1_score(y, p, labels=range(NC), average='macro', zero_division=0)

def describe(y, p):
    rec = {f'recall_{c}': float((p[y == i] == i).mean()) if (y == i).any() else np.nan for i, c in enumerate(CLASSES)}
    return dict(accuracy=float((p == y).mean()), macro_f1=macro_f1(y, p), errors=int((p != y).sum()),
                diseased_called_healthy=int(((y != H_IDX) & (p == H_IDX)).sum()), healthy_called_diseased=int(((y == H_IDX) & (p != H_IDX)).sum()), **rec)

def signflip(net, seed=0):
    net = net[np.abs(net) > 1e-9]
    if not len(net): return 1.0
    sim = (np.random.default_rng(seed).choice([-1, 1], (CFG['n_perm'], len(net))) * net).sum(1)
    return float((1 + np.sum(np.abs(sim) >= abs(net.sum()) - 1e-9)) / (CFG['n_perm'] + 1))

def compare(y, a, b, groups, seed=0):
    # Mean correct over models for each image, then net gain per source photo.
    a, b = np.atleast_2d(a), np.atleast_2d(b)
    oa, ob = (a == y).mean(0), (b == y).mean(0)
    net = pd.Series(ob - oa).groupby(groups).sum().values
    d = np.random.default_rng(seed).choice(net, (CFG['n_boot'], len(net))).sum(1) / len(y)
    return dict(reference_accuracy=float(oa.mean()), accuracy=float(ob.mean()), net_correct=float(net.sum()), d_lo=float(np.percentile(d, 2.5)), d_hi=float(np.percentile(d, 97.5)), p_source=signflip(net, seed))

def holm(ps):
    order = np.argsort(ps); out, run = np.zeros(len(ps)), 0.0
    for r, i in enumerate(order):
        run = max(run, (len(ps) - r) * ps[i]); out[i] = min(1.0, run)
    return out

rows = []
for arm, (oof, h, src) in ARMS.items():
    rows.append(dict(arm=arm, test='development', **compare(Y, OLD_D, oof.argmax(1), GROUPS)))
    rows.append(dict(arm=arm, test='hold-out', **compare(Y_H, OLD_H, h.argmax(2), GROUPS_H)))
    rows.append(dict(arm=arm, test='new source (named to r6k)', **compare(Y_R, SRC_RX['test'].argmax(1), src.argmax(1), GROUPS_R)))
COMPARE = pd.DataFrame(rows)
COMPARE['p_holm'] = np.nan
for t in ('development', 'new source (named to r6k)'):
    m = COMPARE.test == t
    COMPARE.loc[m, 'p_holm'] = holm(COMPARE.loc[m, 'p_source'].values)
COMPARE.to_csv(f'{OUT}/fresh_compare.csv', index=False)

res = []
for name, y, sets in (('development', Y, {'Flat ResNeXt-50': OLD_D, **{a: v[0].argmax(1) for a, v in ARMS.items()}}),
                      ('new source (named to r6k)', Y_R, {'ResNeXt-50 trained on named': SRC_RX['test'].argmax(1), **{a: v[2].argmax(1) for a, v in ARMS.items()}})):
    res += [dict(arm=a, test=name, **describe(y, p)) for a, p in sets.items()]
for a, p in {'Flat ResNeXt-50': OLD_H, **{x: v[1].argmax(2) for x, v in ARMS.items()}}.items():
    d = [describe(Y_H, p[k]) for k in range(K)]
    res.append(dict(arm=a, test='hold-out, mean of 3', **{m: float(np.mean([x[m] for x in d])) for m in d[0]}))
RESULTS = pd.DataFrame(res)
RESULTS.to_csv(f'{OUT}/fresh_results.csv', index=False)

FOLDS = pd.DataFrame([dict(arm=a, fold=k, accuracy=float((v[0].argmax(1)[IDX[k]] == Y[IDX[k]]).mean()), net_vs_flat=int((OLD_D[IDX[k]] != Y[IDX[k]]).sum() - (v[0].argmax(1)[IDX[k]] != Y[IDX[k]]).sum())) for a, v in ARMS.items() for k in range(K)])
FOLDS.to_csv(f'{OUT}/fresh_folds.csv', index=False)

# Does DINOv2 make different mistakes?
OLD_E, HIER_E = OLD_D != Y, dev_df.hier.map(CLASSES.index).values != Y
SHARED_OLD = OLD_E & HIER_E & (dev_df.flat.values == dev_df.hier.values)
OVER = pd.DataFrame([dict(arm=a, errors=int((v[0].argmax(1) != Y).sum()), shared_with_flat=int(((v[0].argmax(1) != Y) & OLD_E).sum()),
                          old_shared_errors=int(SHARED_OLD.sum()), old_shared_errors_fixed=int((v[0].argmax(1)[SHARED_OLD] == Y[SHARED_OLD]).sum())) for a, v in ARMS.items()])
OVER.to_csv(f'{OUT}/fresh_overlap.csv', index=False)

BY_SOURCE = pd.DataFrame([dict(arm=a, source=s, images=int((dev_df.source == s).sum()), accuracy=float((p[(dev_df.source == s).values] == Y[(dev_df.source == s).values]).mean()))
                          for a, p in {'Flat ResNeXt-50': OLD_D, **{x: v[0].argmax(1) for x, v in ARMS.items()}}.items() for s in dev_df.source.unique()])
BY_SOURCE.to_csv(f'{OUT}/fresh_by_source.csv', index=False)

old_row = RESULTS[(RESULTS.arm == 'Flat ResNeXt-50') & (RESULTS.test == 'development')].iloc[0]
checks = {}
for a in ARMS:
    g = lambda t: COMPARE[(COMPARE.arm == a) & (COMPARE.test == t)].iloc[0]
    row = RESULTS[(RESULTS.arm == a) & (RESULTS.test == 'development')].iloc[0]
    acc = {'fewer development errors, Holm p < 0.05': bool(g('development').net_correct > 0 and g('development').p_holm < CFG['rule_p']),
           'no fold worse': bool((FOLDS[FOLDS.arm == a].net_vs_flat >= 0).all()),
           'fewer hold-out errors': bool(g('hold-out').net_correct > 0),
           'no more diseased called healthy': bool(row.diseased_called_healthy <= old_row.diseased_called_healthy)}
    checks[a] = dict(accuracy_checks=acc, better_accuracy=all(acc.values()),
                     better_new_source=bool(g('new source (named to r6k)').net_correct > 0 and g('new source (named to r6k)').p_holm < CFG['rule_p']))

display(RESULTS[['arm', 'test', 'accuracy', 'macro_f1', 'errors', 'diseased_called_healthy']].round(4))
display(COMPARE.round(4))
display(FOLDS.pivot(index='arm', columns='fold', values='net_vs_flat'))
display(OVER); display(BY_SOURCE.round(4))
display(pd.DataFrame({a: {k: v[k] for k in ('better_accuracy', 'better_new_source')} for a, v in checks.items()}).T)

In [ ]:
pd.DataFrame(HISTORIES).to_csv(f'{OUT}/fresh_histories.csv', index=False)
times = {f'fine-tune fold {k}': float(r['train_time']) / 60 for k, r in enumerate(FT)}
times.update({'new source ResNeXt-50': float(SRC_RX['train_time']) / 60, 'new source fine-tune': float(SRC_FT['train_time']) / 60})
json.dump(dict(checks=checks, linear=LIN_INFO, val_sizes=VAL_SIZES, train_minutes=times, session_minutes=(time.time() - SESSION_START) / 60, smoke=SMOKE),
          open(f'{OUT}/fresh_checks.json', 'w'), indent=1, default=float)

dp = dev_df[['file', 'label', 'group', 'fold', 'source', 'flat', 'hier']].copy()
dp['dino_frozen'] = [CLASSES[i] for i in LIN_OOF.argmax(1)]; dp['dino_finetuned'] = [CLASSES[i] for i in FT_OOF.argmax(1)]
dp.to_csv(f'{OUT}/fresh_predictions_dev.csv', index=False)
hp = hold_df[['file', 'label', 'group', 'source']].copy()
for k in range(K):
    hp[f'frozen_m{k}'] = [CLASSES[i] for i in LIN_H[k].argmax(1)]; hp[f'finetuned_m{k}'] = [CLASSES[i] for i in FT_H[k].argmax(1)]
hp.to_csv(f'{OUT}/fresh_predictions_hold.csv', index=False)
rp = R6K[['file', 'label', 'group']].copy()
rp['resnext'] = [CLASSES[i] for i in SRC_RX['test'].argmax(1)]; rp['frozen'] = [CLASSES[i] for i in SRC_LIN.argmax(1)]; rp['finetuned'] = [CLASSES[i] for i in SRC_FT['test'].argmax(1)]
rp.to_csv(f'{OUT}/fresh_predictions_new_source.csv', index=False)
print(f'session time: {(time.time() - SESSION_START) / 60:.1f} min')

## 8. Export results for review

In [ ]:
ZIP = f'{BASE}/fresh_dino_results_for_review.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)):
        if fn.endswith(('.csv', '.json')):
            z.write(f'{OUT}/{fn}', fn)
print('saved', ZIP, round(os.path.getsize(ZIP) / 1e6, 2), 'MB')
if IN_KAGGLE:
    from IPython.display import FileLink
    display(FileLink(os.path.relpath(ZIP)))
    print('If the link does not work, download the zip from the Output panel.')
elif files:
    files.download(ZIP)